<a href="https://colab.research.google.com/github/1kaiser/1kaiser.github.io/blob/main/wmo_385_2012.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -nc https://github.com/1kaiser/R_e/releases/download/1/wmo_385-2012.pdf

--2025-12-09 05:55:55--  https://github.com/1kaiser/R_e/releases/download/1/wmo_385-2012.pdf
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/713043214/b5fb8e08-39b8-408f-9f2e-b472be3b11cf?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-09T06%3A29%3A45Z&rscd=attachment%3B+filename%3Dwmo_385-2012.pdf&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-09T05%3A28%3A48Z&ske=2025-12-09T06%3A29%3A45Z&sks=b&skv=2018-11-09&sig=Q%2BL%2FK%2BsiF6wtmJStmWPb9T%2FvAj6S%2BYcsk6uzngaXlic%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NTI2MDA1NSwibmJmIjoxNzY1MjU5NzU1LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9

In [ ]:
!uv pip install PyMuPDF
import tqdm
import fitz  # PyMuPDF
import os
import json  # Import the json library
import re

pdf_path = "/content/wmo_385-2012.pdf"  # Make sure you have uploaded this file

# --- DEFINE YOUR PAGE RANGE HERE ---
start_page = 0  + 10 # This corresponds to page 5
end_page = 390   + 10 # This corresponds to page 10
# -----------------------------------

# Initialize an empty list to store all the blocks
all_blocks = []
# Regex pattern to split the text into blocks
pattern = r"\n(?=\d{2} \n)"

# Check if the PDF file exists
if not os.path.exists(pdf_path):
    print(f"ERROR: PDF file not found at '{pdf_path}'. Please upload it first.")
else:
    doc = fitz.open(pdf_path)

    # Validate the page range
    if start_page >= 0 and end_page < doc.page_count and start_page <= end_page:
        print(f"--- Extracting text from pages {start_page + 1} to {end_page + 1} ---")

        # Loop through the specified pages
        for page_num in tqdm.tqdm(range(start_page, end_page + 1)):
            page = doc.load_page(page_num)  # Load the current page
            text = page.get_text()         # Extract text from the page
            # Split the text from the current page into blocks
            blocks = re.split(pattern, text)

            # Add the blocks from this page to the main list
            all_blocks.extend(blocks)

            # print(f"\n{'='*15} CONTENT OF PAGE {page_num + 1} {'='*15}\n")
            # print(text)
    else:
        print(f"ERROR: Invalid page range. The document has {doc.page_count} pages.")

    doc.close()

    # --- Save the extracted blocks to a JSON file ---
    output_json_path = "extracted_blocks.json"
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(all_blocks, f, ensure_ascii=False, indent=4) # Use indent for readability

    print(f"\n--- All Blocks Saved to JSON ---")
    print(f"Data saved to: '{output_json_path}'")
    # ------------------------------------------------

# You can now access all the blocks from the `all_blocks` list
print("\n--- All Blocks Appended to a List ---")
# This will print the list of all the blocks extracted from the PDF.
# print(all_blocks) # Uncomment if you still want to print the list to console

Using Python 3.12.12 environment at: /usr
Resolved 1 package in 181ms
Prepared 1 package in 441ms
Installed 1 package in 10ms
 + pymupdf==1.26.6
--- Extracting text from pages 11 to 401 ---


100%|██████████| 391/391 [00:03<00:00, 116.01it/s]


--- All Blocks Saved to JSON ---
Data saved to: 'extracted_blocks.json'

--- All Blocks Appended to a List ---


In [ ]:
import json
data = json.load(open("/content/extracted_blocks.json"))

In [ ]:
data[10]

'09 \naccelerated flow \nFlow in which the velocity increases in the \ndirection of flow. \nécoulement accéléré \nÉcoulement dont la vitesse croît dans le sens du \ncourant. \n \nflujo acelerado \nFlujo en el que la velocidad aumenta en la \ndirección del movimiento. \nускоренное течение \nПоток, в котором скорость увеличивается в \nнаправлении течения. \n \n \n \n \n \n \n'

### regex filtering

In [ ]:
import json
data = json.load(open("/content/extracted_blocks.json"))
extracted_text = "".join(data[1:])
import re
text_entries = re.split(r'\n(?=\d+\s)', extracted_text)

list_of_results = map(lambda t: re.split(r'\s(?=\d{2}\s)', t.strip()), text_entries)
from itertools import chain
final_result = list(chain.from_iterable(list_of_results))

def merge_sequential_chunks(data: list[str]) -> list[str]:
    if not data: return []

    out = [data[0]]
    for item in data[1:]:
        # Check if the current item is sequential to the last item in our output list
        is_sequential = (m := re.match(r'^\s*(\d+)', item)) and \
                        (p := re.match(r'^\s*(\d+)', out[-1])) and \
                        int(m.group(1)) == int(p.group(1)) + 1

        # Append if sequential, otherwise merge the item with the last one
        if is_sequential:
            out.append(item)
        else:
            out[-1] = f"{out[-1].rstrip()} {item.lstrip()}"

    return out


corrected_final_result = merge_sequential_chunks(final_result)

In [ ]:
len(corrected_final_result)

1692

In [ ]:
corrected_final_result[130:135]

["131 \nbase flow syn. base runoff \nDischarge which enters a stream channel mainly \nfrom groundwater, but also from lakes and \nglaciers, \nduring \nlong \nperiods \nwhen \nno \nprecipitation or snowmelt occurs. \ndébit de base  \nPartie du débit d'un cours d'eau qui provient \nessentiellement des nappes souterraines, mais \naussi des lacs et des glaciers, durant les longues \npériodes sans précipitation ni fonte de neige. \n \ncaudal de base sin. escorrentía de base \nCaudal que se incorpora a una corriente de \nagua, procedente principalmente de aguas \nsubterráneas aunque también de lagos y \nglaciares, durante períodos largos en los que no \nse produce ni precipitación ni fusión de nieve. \nбазисный сток \nСток, который поступает в речное русло в \nосновном из подземных вод, а также озер и \nледников в периоды длительного отсутствия \nосадков или снеготаяния.",
 "132 \nbase-width (of a flood hydrograph) \nTime interval between the beginning and the \nend of the direct runoff prod

### ollama based filtering where regex may fail

In [ ]:
# @title Install, Run, and Test Ollama on CPU (All-in-One)
variable_name = ""

import os
import subprocess
import time

# --- 1. Install Ollama ---
print("--- 1. Installing Ollama... ---")
# Using a subprocess to run the installation script
try:
    subprocess.run("curl -fsSL# https://ollama.com/install.sh | sh", shell=True, check=True, capture_output=True, text=True)
    print("Ollama installation complete.")
except subprocess.CalledProcessError as e:
    print("Ollama is likely already installed. Continuing...")
    # Print the stderr to see the actual message, which is often harmless
    print(e.stderr)

# --- 2. Install Ollama Python Library ---
print("\n--- 2. Installing Ollama Python library... ---")
try:
    import ollama
except ImportError:
    subprocess.run("pip install -q ollama", shell=True, check=True)
    import ollama
print("Ollama library is ready.")

# --- 3. Force CPU Mode and Start Server ---
# print("\n--- 3. Forcing CPU mode and starting Ollama server... ---")
# os.environ['CUDA_VISIBLE_DEVICES'] = ""
# Use Popen to start the server as a non-blocking background process
server_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print(f"Ollama server started with PID: {server_process.pid}. Waiting for it to initialize...")
# Give the server a generous amount of time to start up.
time.sleep(10)

# --- 4. Pull the Model and Run the Test ---
model_to_test = "granite4:1b" # @param ["qwen3-vl:8b","granite4:1b"] {"allow-input":true}

try:
    print(f"\n--- 4. Pulling the {model_to_test} model (if not present)... ---")
    ollama.pull(model_to_test)
    print("Model pull complete.")

    print(f"\n--- 5. Sending a test prompt to {model_to_test}... ---")
    stream = ollama.chat(
        model=model_to_test,
        messages=[{'role': 'user', 'content': 'Write a short, simple poem about a robot.'}],
        stream=True,
    )

    print("\n--- Model Response: ---")
    for chunk in stream:
        print(chunk['message']['content'], end='', flush=True)
    print()

except Exception as e:
    print(f"\nAn error occurred: {e}")
    print("--- Server Logs ---")
    # If something went wrong, let's see the server's error log
    stdout, stderr = server_process.communicate()
    print("STDOUT:", stdout.decode())
    print("STDERR:", stderr.decode())

finally:
    # --- 6. Clean up the server process ---
    print("\n--- 6. Shutting down the Ollama server. ---")
    server_process.terminate()
    server_process.wait() # Wait for the process to truly finish
    print("Server has been shut down.")

--- 1. Installing Ollama... ---
Ollama installation complete.

--- 2. Installing Ollama Python library... ---
Ollama library is ready.
Ollama server started with PID: 576. Waiting for it to initialize...

--- 4. Pulling the granite4:1b model (if not present)... ---
Model pull complete.

--- 5. Sending a test prompt to granite4:1b... ---

--- Model Response: ---
In circuits and wires of steel bright,
A robot stands with silent might.
With gears turning in rhythmic flow,
Its purpose to serve, day or night.

With eyes that see beyond sight,
It learns from the world's soft light.
In tasks it performs with precision clear,
As steady as the morning near.

--- 6. Shutting down the Ollama server. ---
Server has been shut down.


In [ ]:
!pip install -q ollama

import time, subprocess, ollama

def start_ollama_server():
    """Start Ollama in background if not running."""
    try:
        ollama.list()
    except Exception:
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(6)

def get_gpu_response(prompt, model='granite4:1b'):
    start_ollama_server()
    response = ollama.chat(model=model, messages=[{'role': 'user', 'content': prompt}])
    if hasattr(response, 'message') and hasattr(response.message, 'content'):
        return response.message.content
    elif isinstance(response, dict) and 'message' in response and 'content' in response['message']:
        return response['message']['content']
    return "⚠️ No valid response from model."




In [ ]:
# --- Your text data ---
text_data = corrected_final_result[341]

# --- Define task ---
model_to_test = "granite4:1b"
prompt = f"""
          From the following multilingual dictionary-style text, extract:
          1. The initial number (e.g. "09")
          2. The English term immediately following it (e.g. "accelerated flow")
          3. The English definition paragraph that follows it (until another language starts)

          Output strictly as a Python dictionary in this format:
          {{
            "09": {{
              "term": "accelerated flow",
              "definition": "Flow in which the velocity increases in the direction of flow."
            }}
          }}

          Text:
          {text_data}
          """

# --- Run ---
print(get_gpu_response(prompt, model=model_to_test))

{
  "342": {
    "term": "decay rate",
    "definition": "1) Rate of reduction of the concentration of a substance. \n2) Rate of reduction of the activity of a radioactive isotope, expressed in terms of half-life."
  }
}


In [ ]:
corrected_final_result[341]

"342 \ndecay rate see also half-life \n(1) Rate of reduction of the concentration of a \nsubstance. \n(2) Rate of reduction of the activity of a \nradioactive isotope, expressed in terms of half-\nlife. \ntaux de décroissance voir aussi demie-vie \n1) Taux de réduction de la concentration d'une \nsubstance. \n2) Taux de réduction de l'activité d'un radio-\nisotope exprimé par sa période. \n \nvelocidad de desintegración véase también \nperíodo de semidesintegración \n1) Tasa de reducción de la concentración de \nuna sustancia. \n2) Tasa de reducción de la actividad de un \nisótopo radiactivo expresada en términos de su \nperíodo de semidesintegración. \nскорость \nраспада \nсм. \nтакже \nпериод \nполураспада \n(1) \nСкорость \nуменьшения \nконцентрации \nвещества. \n(2) \nСкорость \nснижения \nактивности \nрадиоактивного \nизотопа, \nвыраженная \nпериодом полураспада."

In [ ]:
# @title ###--- Iterate through all text entries with tqdm ---
import tqdm, ast, json

merged_dict = {}

# --- Main Loop ---
for i, text_data in enumerate(tqdm.tqdm(corrected_final_result, desc="Processing entries")):
    prompt = f"""
    From the following multilingual dictionary-style text, extract:
    1. The initial number (e.g. "09")
    2. The English term immediately following it (e.g. "accelerated flow")
    3. The English definition paragraph that follows it (until another language starts)

    Output strictly as a Python dictionary in this format:
    {{
      "09": {{
        "term": "accelerated flow",
        "definition": "Flow in which the velocity increases in the direction of flow."
      }}
    }}

    Text:
    {text_data}
    """

    try:
        response = get_gpu_response(prompt, model=model_to_test)
        try:
            parsed = ast.literal_eval(response.strip())
            if isinstance(parsed, dict):
                merged_dict.update(parsed)
            else:
                merged_dict[str(i)] = {"raw_output": response}
        except Exception:
            merged_dict[str(i)] = {"raw_output": response}
    except Exception as e:
        merged_dict[str(i)] = {"error": str(e)}

# --- Save final merged results ---
output_path = "/content/dictionary_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(merged_dict, f, indent=2, ensure_ascii=False)

print(f"\n✅ Completed! Merged dictionary saved to {output_path}")
print(f"✅ Total parsed entries: {len(merged_dict)}")


Processing entries: 100%|██████████| 1692/1692 [37:32<00:00,  1.33s/it]


✅ Completed! Merged dictionary saved to /content/dictionary_results.json
✅ Total parsed entries: 1654


In [ ]:
import json
dictionary_results = json.load(open("/content/dictionary_results.json"))
dictionary_results = {key: value for key, value in dictionary_results.items() if key.isdigit()}


In [ ]:
len(list(dictionary_results)), dictionary_results.keys()

(1606,
 dict_keys(['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '110', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '

In [ ]:
def find_and_add_missing_keys(data_dict: dict) -> list[int]:
    """
    Finds missing sequential numbers in dictionary keys, and adds them back.

    Args:
        data_dict: The dictionary to process. It will be modified in-place.

    Returns:
        A list of the missing numbers that were added.
    """
    # 1. Extract all keys that are purely digits and convert them to integers
    numeric_keys = [int(key) for key in data_dict.keys() if key.isdigit()]

    if not numeric_keys:
        print("No numeric keys found in the dictionary.")
        return []

    # 2. Determine the full, expected sequence and find what's missing
    min_val = min(numeric_keys)
    max_val = 1692

    # Create a set of the full range of numbers for efficient lookup
    full_sequence = set(range(min_val, max_val + 1))

    # Find the difference between the full set and the actual keys
    missing_numbers = sorted(list(full_sequence - set(numeric_keys)))

    # 3. Add the missing keys back into the dictionary
    if missing_numbers:
        print(f"Found missing numbers: {missing_numbers}")
        # Determine the correct string length for zero-padding (e.g., '01' vs '100')
        padding = len(str(max_val))

        for num in missing_numbers:
            # Format the number back to a string with leading zeros if needed
            missing_key = str(num).zfill(padding)
            data_dict[missing_key] = None  # Add with a placeholder value
            print(f"Added missing key: '{missing_key}'")
    else:
        print("No missing numbers found in the sequence.")

    return missing_numbers

# --- Example Usage ---

# Your dictionary with some non-numeric keys and missing numbers (like '03', '05')
my_dictionary = dictionary_results

print("Original keys:", sorted(my_dictionary.keys()))
print("-" * 20)

# Run the function to find and add the missing keys
find_and_add_missing_keys(my_dictionary)

print("-" * 20)
# Print the updated keys, sorted to show the result clearly
print("Final keys after adding missing ones:", sorted(my_dictionary.keys()))

Original keys: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '100', '1000', '1001', '1002', '1003', '1004', '1005', '1006', '1007', '1008', '1009', '101', '1010', '1011', '1012', '1013', '1014', '1015', '1016', '1017', '1018', '1019', '102', '1020', '1021', '1022', '1023', '1024', '1025', '1026', '1027', '1028', '103', '1030', '1031', '1032', '1033', '1034', '1035', '1036', '1037', '1038', '1039', '104', '1040', '1042', '1043', '1044', '1045', '1046', '1047', '1048', '1049', '105', '1050', '1051', '1052', '1053', '1054', '1055', '1056', '1057', '1058', '1059', '106', '1060', '1061', '1062', '1063', '1064', '1065', '1066', '1067', '1068', '1069', '107', '1070', '1071', '1072', '1073', '1074', '1075', '1076', '1077', '1078', '1079', '108', '1080', '1081', '1082', '1083', '1084', '1085', '1086', '1087', '1088', '1089', '1091', '1092', '1093', '1094', '1095', '1096', '1097', '1098', '1099', '11', '110', '1100', '1101', '1102', '1103', '1104', '1105', '1106', '1107', '1108', 

In [ ]:
my_dictionary['10']

{'term': 'acceptance capacity',
 'definition': 'Quantity of pollutants which a water body can accept without the pollution exceeding a given level.'}

In [ ]:

# --- Save final merged results ---
output_path = "/content/dictionary.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(my_dictionary, f, indent=2, ensure_ascii=False)

In [ ]:
# --- Your text data ---
text_data = corrected_final_result[342]

# --- Define task ---
model_to_test = "granite4:1b"
prompt = f"""
          From the following multilingual dictionary-style text, extract:
          1. The initial number (e.g. "09")
          2. The English term immediately following it (e.g. "accelerated flow")
          3. The English definition paragraph that follows it (until another language starts)

          Output strictly as a Python dictionary in this format:
          {{
            "09": {{
              "term": "accelerated flow",
              "definition": "Flow in which the velocity increases in the direction of flow."
            }}
          }}

          Text:
          {text_data}
          """

# --- Run ---
print(get_gpu_response(prompt, model=model_to_test))

{
  "03": {
    "term": "choice",
    "definition": "A decision between alternatives."
  }
}


In [ ]:
# 1. Install the correct package (umap-learn)
# 2. Pin NumPy to version 1.x to avoid compatibility issues with Numba/SciPy/UMAP
!pip install "numpy<2.0" "scipy>=1.10" "umap-learn" "sentence-transformers" plotly

# -----------------------------------------------------------------------
# IMPORTANT: If prompted, click "Restart Runtime" or "Restart Session"
# button in the output below this cell for changes to take effect.
# -----------------------------------------------------------------------


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 14.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.5
    Uninstalling numpy-2.3.5:
      Successfully uninstalled numpy-2.3.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which 

In [ ]:
import json
import pandas as pd
import umap  # Now this will work
import plotly.express as px
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# 1. INPUT DATA
# ---------------------------------------------------------
file_path = "/content/dictionary.json"

try:
    with open(file_path, 'r') as f:
        data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    data = {}

# # ---------------------------------------------------------
# # 2. EXTRACT DATA TO LISTS
# # ---------------------------------------------------------
# terms = []
# definitions = []
# ids = []

# print(f"Processing {len(data)} items...")

# for key, item in data.items():
#     if not isinstance(item, dict):
#         continue

#     # Extract term and definition
#     term_val = item.get('term') or item.get('Term') or item.get('word') or item.get('Word')
#     def_val = item.get('definition') or item.get('Definition') or item.get('desc') or item.get('meaning')

#     if term_val:
#         terms.append(term_val)
#         definitions.append(def_val if def_val else "No definition found")
#         ids.append(key)

# print(f"Successfully extracted {len(terms)} valid terms.")

# if len(terms) == 0:
#     # Handle empty data gracefully instead of crashing
#     print("Warning: No terms found. Using dummy data for demonstration.")
#     terms = ["River", "Aquifer", "Drought"]

# ---------------------------------------------------------
# 2. EXTRACT DATA TO LISTS
# ---------------------------------------------------------
terms = []
definitions = []
ids = []

print(f"Processing {len(data)} items...")

seen_terms = set()  # LINE 1

for key, item in data.items():
    if not isinstance(item, dict):
        continue

    # Extract term and definition
    term_val = item.get('term') or item.get('Term') or item.get('word') or item.get('Word')
    def_val = item.get('definition') or item.get('Definition') or item.get('desc') or item.get('meaning')

    if term_val and term_val.lower().strip() not in seen_terms and not seen_terms.add(term_val.lower().strip()):  # LINE 2
        terms.append(term_val)
        definitions.append(def_val if def_val else "No definition found")
        ids.append(key)

print(f"Successfully extracted {len(terms)} valid terms.")

if len(terms) == 0:
    # Handle empty data gracefully instead of crashing
    print("Warning: No terms found. Using dummy data for demonstration.")
    terms = ["River", "Aquifer", "Drought"]


# ---------------------------------------------------------
# 3. GENERATE EMBEDDINGS (EmbeddingGemma)
# ---------------------------------------------------------
print("Loading EmbeddingGemma model...")
model = SentenceTransformer('google/embeddinggemma-300m', truncate_dim=256)

print("Generating embeddings...")
embeddings = model.encode(
    terms,
    prompt_name="Clustering",
    normalize_embeddings=True,
    show_progress_bar=True
)
# ---------------------------------------------------------
# 4. UMAP REDUCTION (3D)
# ---------------------------------------------------------
print("Running UMAP...")

# Dynamic neighbors: UMAP needs n_neighbors < n_samples
n_neighbors = min(len(terms) - 1, 15)
if n_neighbors < 2: n_neighbors = 2

reducer = umap.UMAP(
    n_components=3,
    n_neighbors=n_neighbors,
    min_dist=0.1,
    random_state=42
)
projections = reducer.fit_transform(embeddings)




Processing 1692 items...
Successfully extracted 1472 valid terms.
Loading EmbeddingGemma model...
Generating embeddings...


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Running UMAP...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
# ---------------------------------------------------------
# @title # 5. CREATE DATAFRAME & PLOT WITH PLOTLY (FIXED)
# ---------------------------------------------------------
print("Creating Plot...")

df = pd.DataFrame(projections, columns=['x', 'y', 'z'])
df['term'] = terms
df['definition'] = definitions
df['id'] = ids

# Create 3D Scatter Plot
fig = px.scatter_3d(
    df,
    x='x',
    y='y',
    z='z',
    hover_name='term',
    hover_data={
        'x': False,
        'y': False,
        'z': False,
        'definition': True,
        'term': False
    },
    title=f"3D UMAP Word Embeddings ({len(terms)} items)"
)

# Update markers
fig.update_traces(
    marker=dict(
        size=6,
        opacity=0.7,
        color='steelblue',
        line=dict(width=0.5, color='white')
    )
)

# Update layout with proper window configuration
fig.update_layout(
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=40),
    scene=dict(
        xaxis_title='UMAP Dimension 1',
        yaxis_title='UMAP Dimension 2',
        zaxis_title='UMAP Dimension 3',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
    ),
    showlegend=False
)

fig.show()


Creating Plot...


In [ ]:
from IPython.display import HTML, display
import json
import numpy as np


def create_d3_scatter_infinite_zoom(projections, terms, definitions, ids, height=850):
    """
    D3.js 3D scatter plot with infinite, smooth exponential zooming and orbital controls.
    """

    # Clean Data
    clean_proj = np.nan_to_num(projections, nan=0.0, posinf=0.0, neginf=0.0)

    data = []
    for i in range(len(terms)):
        data.append({
            'x': float(clean_proj[i, 0]),
            'y': float(clean_proj[i, 1]),
            'z': float(clean_proj[i, 2]),
            'term': str(terms[i]).replace('"', '\\"'),
            'definition': (str(definitions[i])[:120] + '...').replace('"', '\\"'),
            'id': str(ids[i])
        })

    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <script src="https://d3js.org/d3.v7.min.js"></script>
        <style>
            body, html {{ margin: 0; padding: 0; overflow: hidden; }}

            #viz-wrapper {{
                position: relative;
                width: 100%;
                height: {height}px;
                background: radial-gradient(circle at center, #f8f9fa 0%, #e0e7ff 100%);
                border: 1px solid #ddd;
                border-radius: 8px;
                cursor: default;
            }}

            #main-canvas {{
                display: block;
                width: 100%;
                height: 100%;
            }}

            .overlay {{
                position: absolute;
                top: 20px;
                left: 20px;
                pointer-events: none;
                z-index: 10;
                background: rgba(255, 255, 255, 0.8);
                padding: 10px 15px;
                border-radius: 8px;
                backdrop-filter: blur(4px);
                border: 1px solid rgba(0,0,0,0.05);
            }}

            .title {{
                font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
                font-size: 18px;
                font-weight: 700;
                color: #2c3e50;
                margin: 0;
            }}

            .subtitle {{
                font-family: sans-serif;
                font-size: 12px;
                color: #666;
                margin-top: 4px;
            }}


            .controls-hint {{
                position: absolute;
                bottom: 20px;
                left: 50%;
                transform: translateX(-50%);
                background: rgba(0,0,0,0.7);
                color: white;
                padding: 8px 16px;
                border-radius: 20px;
                font-size: 12px;
                font-family: sans-serif;
                pointer-events: none;
                opacity: 0.6;
                transition: opacity 0.3s;
            }}

            .tooltip {{
                position: absolute;
                background: rgba(30, 30, 35, 0.95);
                color: #fff;
                padding: 12px 16px;
                border-radius: 8px;
                font-family: sans-serif;
                font-size: 13px;
                line-height: 1.4;
                max-width: 280px;
                pointer-events: none;
                opacity: 0;
                transition: opacity 0.1s;
                z-index: 100;
                box-shadow: 0 10px 25px rgba(0,0,0,0.2);
                transform: translateY(-50%);
            }}

            .term-header {{ font-weight: 700; color: #60a5fa; margin-bottom: 4px; font-size: 14px; }}
        </style>
    </head>
    <body>
        <div id="viz-wrapper">
            <canvas id="main-canvas"></canvas>

            <div class="overlay">
                <div class="title">Semantic Space</div>
                <div class="subtitle">{len(data)} items • Orbital Controls</div>
            </div>

            <div class="controls-hint">
                🖱️ Left: Rotate &nbsp; • &nbsp; 🖱️ Right: Pan &nbsp; • &nbsp; 🖱️ Scroll: Zoom
            </div>

            <div id="tooltip" class="tooltip"></div>
        </div>


        <script>
            const data = {json.dumps(data)};
            const wrapper = document.getElementById('viz-wrapper');
            const canvas = document.getElementById('main-canvas');
            const ctx = canvas.getContext('2d');
            const tooltip = document.getElementById('tooltip');

            // --- CONFIGURATION ---
            // "Camera" state with panning support
            let state = {{
                rotationX: 0.5,
                rotationY: 0.5,
                scale: 1.0,      // Master scale factor
                distance: 2.0,   // Base camera distance
                panX: 0.0,       // Pan offset X
                panY: 0.0        // Pan offset Y
            }};

            // Interaction State
            let isDragging = false;
            let dragButton = null;  // Track which button is being dragged
            let lastPos = {{ x: 0, y: 0 }};
            let width, height;


            // --- SCALING HELPERS ---
            // Normalize data to -1...1 range initially
            const xExt = d3.extent(data, d => d.x);
            const yExt = d3.extent(data, d => d.y);
            const zExt = d3.extent(data, d => d.z);

            const xScale = d3.scaleLinear().domain(xExt).range([-1, 1]);
            const yScale = d3.scaleLinear().domain(yExt).range([-1, 1]);
            const zScale = d3.scaleLinear().domain(zExt).range([-1, 1]);


            // --- RESIZE HANDLER ---
            function resize() {{
                width = wrapper.clientWidth;
                height = wrapper.clientHeight;
                const dpr = window.devicePixelRatio || 1;
                canvas.width = width * dpr;
                canvas.height = height * dpr;
                canvas.style.width = width + 'px';
                canvas.style.height = height + 'px';
                ctx.scale(dpr, dpr);
                requestAnimationFrame(render);
            }}
            window.addEventListener('resize', resize);


            // --- RENDER LOOP ---
            function render() {{
                ctx.clearRect(0, 0, width, height);

                // Pre-calculate rotation matrices
                const cx = Math.cos(state.rotationX);
                const sx = Math.sin(state.rotationX);
                const cy = Math.cos(state.rotationY);
                const sy = Math.sin(state.rotationY);

                // Base spread of the cloud relative to screen size
                const baseSize = Math.min(width, height) * 0.4;

                const projected = [];

                for(let i=0; i<data.length; i++) {{
                    const d = data[i];

                    // 1. Initial Position (Normalized)
                    const x = xScale(d.x);
                    const y = yScale(d.y);
                    const z = zScale(d.z);

                    // 2. Rotation
                    // Rotate around Y
                    const x1 = x * cy + z * sy;
                    const z1 = -x * sy + z * cy;

                    // Rotate around X
                    const y1 = y * cx - z1 * sx;
                    const z2 = y * sx + z1 * cx;

                    // 3. Apply Camera Distance (Perspective)
                    const cameraZ = z2 + state.distance;

                    // If point is behind camera, skip it
                    if (cameraZ <= 0.1) continue;

                    // 4. Perspective Projection with Panning
                    const perspective = 1 / cameraZ;

                    const screenX = width/2 + (x1 * perspective * baseSize * state.scale) + state.panX;
                    const screenY = height/2 - (y1 * perspective * baseSize * state.scale) + state.panY;

                    // Scale point size by zoom
                    const pointScale = perspective * state.scale;

                    projected.push({{
                        x: screenX,
                        y: screenY,
                        z: z2,
                        size: pointScale,
                        original: d
                    }});
                }}

                // Sort by Z (Depth)
                projected.sort((a, b) => b.z - a.z);

                // Draw
                projected.forEach(p => {{
                    let r = Math.max(1.5, 3 * p.size);

                    if (r < 0.5) return;

                    ctx.beginPath();
                    ctx.arc(p.x, p.y, r, 0, Math.PI * 2);

                    const depthAlpha = Math.max(0.2, Math.min(1, (p.z + 1.5) / 2));

                    ctx.fillStyle = `rgba(37, 99, 235, ${{depthAlpha}})`;
                    ctx.fill();
                }});

                canvas.projected = projected;
            }}

            // --- INTERACTION ---

            // Zoom (Wheel)
            wrapper.addEventListener('wheel', e => {{
                e.preventDefault();

                const zoomSpeed = 0.0015;
                const factor = Math.exp(-e.deltaY * zoomSpeed);

                state.scale *= factor;
                state.scale = Math.max(0.1, Math.min(100.0, state.scale));

                requestAnimationFrame(render);
            }}, {{ passive: false }});

            // Mouse Down - Track which button
            canvas.addEventListener('mousedown', e => {{
                e.preventDefault();
                isDragging = true;
                dragButton = e.button;  // 0 = left, 1 = middle, 2 = right
                lastPos = {{ x: e.clientX, y: e.clientY }};

                if (dragButton === 0) {{
                    wrapper.style.cursor = 'grabbing';
                }} else if (dragButton === 2) {{
                    wrapper.style.cursor = 'move';
                }}
            }});

            // Prevent context menu on right-click
            canvas.addEventListener('contextmenu', e => {{
                e.preventDefault();
            }});

            // Mouse Up
            window.addEventListener('mouseup', () => {{
                isDragging = false;
                dragButton = null;
                wrapper.style.cursor = 'default';
            }});

            // Mouse Move
            window.addEventListener('mousemove', e => {{
                if(isDragging) {{
                    const dx = e.clientX - lastPos.x;
                    const dy = e.clientY - lastPos.y;

                    if (dragButton === 0) {{
                        // Left button - Rotate (Orbit)
                        state.rotationY += dx * 0.005;
                        state.rotationX += dy * 0.005;
                    }} else if (dragButton === 2 || dragButton === 1) {{
                        // Right or Middle button - Pan
                        state.panX += dx;
                        state.panY += dy;
                    }}

                    lastPos = {{ x: e.clientX, y: e.clientY }};
                    requestAnimationFrame(render);
                }} else {{
                    // Tooltip Logic
                    const rect = canvas.getBoundingClientRect();
                    const mx = e.clientX - rect.left;
                    const my = e.clientY - rect.top;

                    if (!canvas.projected) return;

                    let hit = null;
                    for (let i = canvas.projected.length - 1; i >= 0; i--) {{
                        const p = canvas.projected[i];
                        const dist = (p.x - mx)**2 + (p.y - my)**2;
                        const hitRadius = Math.max(25, (10 * p.size)**2);

                        if (dist < hitRadius) {{
                            hit = p;
                            break;
                        }}
                    }}

                    if(hit) {{
                        tooltip.style.opacity = 1;
                        tooltip.style.left = (e.clientX + 20) + 'px';
                        tooltip.style.top = e.clientY + 'px';
                        tooltip.innerHTML = `<div class="term-header">${{hit.original.term}}</div>${{hit.original.definition}}`;
                        canvas.style.cursor = 'pointer';
                    }} else {{
                        tooltip.style.opacity = 0;
                        canvas.style.cursor = 'default';
                    }}
                }}
            }});

            // Start
            resize();
        </script>
    </body>
    </html>
    """

    return HTML(html)


# Execute
viz = create_d3_scatter_infinite_zoom(projections, terms, definitions, ids)
display(viz)


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.neighbors import kneighbors_graph
from scipy.spatial import ConvexHull
from scipy.sparse import csr_matrix

def normalize_adjacency(adj_matrix):
    """
    Compute D^-0.5 * A * D^-0.5 for symmetric normalization
    """
    # Add self-loops
    adj = adj_matrix + np.eye(adj_matrix.shape[0])

    # Degree matrix
    degree = np.array(adj.sum(axis=1)).flatten()
    degree_inv_sqrt = np.power(degree, -0.5)
    degree_inv_sqrt[np.isinf(degree_inv_sqrt)] = 0.

    # D^-0.5 * A * D^-0.5
    D_inv_sqrt = np.diag(degree_inv_sqrt)
    adj_normalized = D_inv_sqrt @ adj @ D_inv_sqrt

    return adj_normalized


class SimpleGCN(tf.keras.Model):
    """
    Simple Graph Convolutional Network for clustering
    """
    def __init__(self, hidden_dims, num_clusters):
        super(SimpleGCN, self).__init__()

        self.layers_list = []
        for dim in hidden_dims:
            self.layers_list.append(
                tf.keras.layers.Dense(dim, activation='relu')
            )

        # Clustering output layer
        self.cluster_layer = tf.keras.layers.Dense(num_clusters, activation='softmax')

    def call(self, x, adj_norm, training=False):
        """
        Forward pass: propagate through graph
        """
        h = x
        for layer in self.layers_list:
            # Graph convolution: A_norm * H * W
            h = tf.matmul(adj_norm, h)
            h = layer(h)

        # Cluster assignments
        cluster_probs = self.cluster_layer(h)

        return cluster_probs


def gnn_clustering_workflow(projections, terms, num_clusters=10, k_neighbors=10):
    """
    Complete GNN-based clustering workflow
    """
    n_nodes = len(projections)

    # 1. BUILD K-NN GRAPH
    adj_matrix = kneighbors_graph(
        projections,
        n_neighbors=k_neighbors,
        mode='connectivity',
        include_self=False
    ).toarray()

    # Make symmetric
    adj_matrix = np.maximum(adj_matrix, adj_matrix.T)

    # 2. NORMALIZE ADJACENCY
    adj_normalized = normalize_adjacency(adj_matrix)
    adj_norm_tf = tf.constant(adj_normalized, dtype=tf.float32)

    # 3. PREPARE INPUT FEATURES
    X = tf.constant(projections, dtype=tf.float32)

    # 4. BUILD GCN MODEL
    model = SimpleGCN(
        hidden_dims=[64, 32],
        num_clusters=num_clusters
    )

    # 5. DEFINE LOSS (Modularity + Regularization)
    def clustering_loss(cluster_probs, adj_matrix):
        # Modularity loss
        degrees = tf.reduce_sum(adj_matrix, axis=1, keepdims=True)
        m = tf.reduce_sum(degrees) / 2.0

        # Q = A - (d * d^T) / 2m
        modularity_matrix = adj_matrix - (degrees @ tf.transpose(degrees)) / (2 * m)

        # Maximize trace(S^T @ Q @ S)
        mod_loss = -tf.linalg.trace(
            tf.transpose(cluster_probs) @ modularity_matrix @ cluster_probs
        ) / (2 * m)

        # Orthogonality regularization - FIXED
        cluster_sizes = tf.reduce_sum(cluster_probs, axis=0)
        mean_size = tf.reduce_mean(cluster_sizes)
        size_loss = tf.reduce_mean(tf.square(cluster_sizes - mean_size))

        # Entropy regularization (encourage confident assignments)
        entropy = -tf.reduce_mean(
            tf.reduce_sum(cluster_probs * tf.math.log(cluster_probs + 1e-10), axis=1)
        )

        return mod_loss + 0.1 * size_loss - 0.01 * entropy


    # 6. TRAINING
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

    adj_matrix_tf = tf.constant(adj_matrix, dtype=tf.float32)

    print("Training GNN clustering model...")
    for epoch in range(200):
        with tf.GradientTape() as tape:
            cluster_probs = model(X, adj_norm_tf, training=True)
            loss = clustering_loss(cluster_probs, adj_matrix_tf)

        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.numpy():.4f}")

    # 7. GET CLUSTER ASSIGNMENTS
    final_probs = model(X, adj_norm_tf, training=False)
    cluster_labels = tf.argmax(final_probs, axis=1).numpy()

    print(f"\nClustering complete!")
    print(f"Number of clusters: {len(np.unique(cluster_labels))}")

    # 8. COMPUTE CONVEX HULLS
    hulls = {}
    hull_vertices = {}

    for cluster_id in np.unique(cluster_labels):
        cluster_mask = cluster_labels == cluster_id
        cluster_points = projections[cluster_mask]

        if len(cluster_points) >= 4:
            try:
                hull = ConvexHull(cluster_points)
                hulls[cluster_id] = hull
                hull_vertices[cluster_id] = cluster_points[hull.vertices]
                print(f"Cluster {cluster_id}: {len(cluster_points)} points, "
                      f"volume: {hull.volume:.3f}")
            except:
                pass

    return {
        'labels': cluster_labels,
        'probabilities': final_probs.numpy(),
        'hulls': hulls,
        'hull_vertices': hull_vertices,
        'adjacency': adj_matrix,
        'model': model
    }


# Execute
results = gnn_clustering_workflow(
    projections=projections,
    terms=terms,
    num_clusters=10,
    k_neighbors=10
)

# Access results
cluster_labels = results['labels']
cluster_probs = results['probabilities']
hulls = results['hulls']


Training GNN clustering model...
Epoch 0, Loss: 1995.8573
Epoch 50, Loss: 1.5958
Epoch 100, Loss: -0.0088
Epoch 150, Loss: -0.0230

Clustering complete!
Number of clusters: 6
Cluster 3: 226 points, volume: 8.680
Cluster 4: 80 points, volume: 1.245
Cluster 5: 80 points, volume: 4.202
Cluster 6: 32 points, volume: 0.659
Cluster 7: 244 points, volume: 7.658
Cluster 9: 810 points, volume: 57.996


In [ ]:
from IPython.display import HTML
import json
import numpy as np

def create_gnn_cluster_3d_viz(projections, cluster_labels, hull_vertices, height=850):
    """
    D3.js 3D scatter plot for GNN clustering with convex hull envelopes.
    """

    # Clean Data
    clean_proj = np.nan_to_num(projections, nan=0.0, posinf=0.0, neginf=0.0)

    # Prepare point data
    data = []
    for i in range(len(cluster_labels)):
        data.append({
            'x': float(clean_proj[i, 0]),
            'y': float(clean_proj[i, 1]),
            'z': float(clean_proj[i, 2]),
            'cluster': int(cluster_labels[i]),
            'id': i
        })

    # Prepare hull data
    hulls = []
    for cluster_id, hull_verts in hull_vertices.items():
        hull_clean = np.nan_to_num(hull_verts, nan=0.0, posinf=0.0, neginf=0.0)
        hull_points = []
        for point in hull_clean:
            hull_points.append({
                'x': float(point[0]),
                'y': float(point[1]),
                'z': float(point[2])
            })
        hulls.append({
            'cluster': int(cluster_id),
            'vertices': hull_points
        })

    # Get unique clusters and count
    unique_clusters = list(set([int(label) for label in cluster_labels]))
    n_clusters = len(unique_clusters)

    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <script src="https://d3js.org/d3.v7.min.js"></script>
        <style>
            body, html {{ margin: 0; padding: 0; overflow: hidden; }}

            #viz-wrapper {{
                position: relative;
                width: 100%;
                height: {height}px;
                background: radial-gradient(circle at center, #f8f9fa 0%, #e0e7ff 100%);
                border: 1px solid #ddd;
                border-radius: 8px;
                cursor: default;
            }}

            #main-canvas {{
                display: block;
                width: 100%;
                height: 100%;
            }}

            .overlay {{
                position: absolute;
                top: 20px;
                left: 20px;
                pointer-events: none;
                z-index: 10;
                background: rgba(255, 255, 255, 0.85);
                padding: 12px 18px;
                border-radius: 8px;
                backdrop-filter: blur(4px);
                border: 1px solid rgba(0,0,0,0.05);
            }}

            .title {{
                font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
                font-size: 18px;
                font-weight: 700;
                color: #2c3e50;
                margin: 0;
            }}

            .subtitle {{
                font-family: sans-serif;
                font-size: 12px;
                color: #666;
                margin-top: 4px;
            }}

            .controls-hint {{
                position: absolute;
                bottom: 20px;
                left: 50%;
                transform: translateX(-50%);
                background: rgba(0,0,0,0.7);
                color: white;
                padding: 8px 16px;
                border-radius: 20px;
                font-size: 12px;
                font-family: sans-serif;
                pointer-events: none;
                opacity: 0.6;
            }}

            .tooltip {{
                position: absolute;
                background: rgba(30, 30, 35, 0.95);
                color: #fff;
                padding: 12px 16px;
                border-radius: 8px;
                font-family: sans-serif;
                font-size: 13px;
                line-height: 1.4;
                pointer-events: none;
                opacity: 0;
                transition: opacity 0.1s;
                z-index: 100;
                box-shadow: 0 10px 25px rgba(0,0,0,0.2);
            }}

            .cluster-id {{ font-weight: 700; color: #60a5fa; margin-bottom: 4px; font-size: 14px; }}
        </style>
    </head>
    <body>
        <div id="viz-wrapper">
            <canvas id="main-canvas"></canvas>

            <div class="overlay">
                <div class="title">GNN Clustering (3D)</div>
                <div class="subtitle">{len(data)} points • {n_clusters} clusters</div>
            </div>

            <div class="controls-hint">
                🖱️ Left: Rotate &nbsp; • &nbsp; 🖱️ Right: Pan &nbsp; • &nbsp; 🖱️ Scroll: Zoom
            </div>

            <div id="tooltip" class="tooltip"></div>
        </div>

        <script>
            const data = {json.dumps(data)};
            const hulls = {json.dumps(hulls)};

            const wrapper = document.getElementById('viz-wrapper');
            const canvas = document.getElementById('main-canvas');
            const ctx = canvas.getContext('2d');
            const tooltip = document.getElementById('tooltip');

            // Cluster Colors (10 distinct colors)
            const clusterColors = [
                [59, 130, 246],   // blue
                [239, 68, 68],    // red
                [34, 197, 94],    // green
                [234, 179, 8],    // yellow
                [168, 85, 247],   // purple
                [236, 72, 153],   // pink
                [20, 184, 166],   // teal
                [249, 115, 22],   // orange
                [14, 165, 233],   // sky
                [132, 204, 22]    // lime
            ];

            // State
            let state = {{
                rotationX: 0.5,
                rotationY: 0.5,
                scale: 1.0,
                distance: 2.5,
                panX: 0.0,
                panY: 0.0
            }};

            let isDragging = false;
            let dragButton = null;
            let lastPos = {{ x: 0, y: 0 }};
            let width, height;

            // Normalize data
            const xExt = d3.extent(data, d => d.x);
            const yExt = d3.extent(data, d => d.y);
            const zExt = d3.extent(data, d => d.z);

            const xScale = d3.scaleLinear().domain(xExt).range([-1, 1]);
            const yScale = d3.scaleLinear().domain(yExt).range([-1, 1]);
            const zScale = d3.scaleLinear().domain(zExt).range([-1, 1]);

            function resize() {{
                width = wrapper.clientWidth;
                height = wrapper.clientHeight;
                const dpr = window.devicePixelRatio || 1;
                canvas.width = width * dpr;
                canvas.height = height * dpr;
                canvas.style.width = width + 'px';
                canvas.style.height = height + 'px';
                ctx.scale(dpr, dpr);
                requestAnimationFrame(render);
            }}
            window.addEventListener('resize', resize);

            // Project 3D point to 2D
            function project3D(x, y, z) {{
                // 1. Rotation
                const cx = Math.cos(state.rotationX);
                const sx = Math.sin(state.rotationX);
                const cy = Math.cos(state.rotationY);
                const sy = Math.sin(state.rotationY);

                const x1 = x * cy + z * sy;
                const z1 = -x * sy + z * cy;

                const y1 = y * cx - z1 * sx;
                const z2 = y * sx + z1 * cx;

                // 2. Camera distance
                const cameraZ = z2 + state.distance;
                if (cameraZ <= 0.1) return null;

                // 3. Perspective projection
                const baseSize = Math.min(width, height) * 0.4;
                const perspective = 1 / cameraZ;

                const screenX = width/2 + (x1 * perspective * baseSize * state.scale) + state.panX;
                const screenY = height/2 - (y1 * perspective * baseSize * state.scale) + state.panY;

                return {{ x: screenX, y: screenY, z: z2, scale: perspective * state.scale }};
            }}

            function render() {{
                ctx.clearRect(0, 0, width, height);

                const projected = [];
                const hullProjected = [];

                // Project points
                for(let i=0; i<data.length; i++) {{
                    const d = data[i];
                    const x = xScale(d.x);
                    const y = yScale(d.y);
                    const z = zScale(d.z);

                    const p = project3D(x, y, z);
                    if (!p) continue;

                    projected.push({{
                        ...p,
                        cluster: d.cluster,
                        original: d
                    }});
                }}

                // Project hulls
                hulls.forEach(hull => {{
                    const hullVerts = [];
                    hull.vertices.forEach(v => {{
                        const x = xScale(v.x);
                        const y = yScale(v.y);
                        const z = zScale(v.z);
                        const p = project3D(x, y, z);
                        if (p) hullVerts.push(p);
                    }});

                    if (hullVerts.length >= 3) {{
                        // Compute average depth for sorting
                        const avgZ = hullVerts.reduce((sum, v) => sum + v.z, 0) / hullVerts.length;
                        hullProjected.push({{
                            cluster: hull.cluster,
                            vertices: hullVerts,
                            avgZ: avgZ
                        }});
                    }}
                }});

                // Sort by depth (back to front)
                projected.sort((a, b) => b.z - a.z);
                hullProjected.sort((a, b) => b.avgZ - a.avgZ);

                // Draw hulls first (background)
                hullProjected.forEach(hull => {{
                    const color = clusterColors[hull.cluster % clusterColors.length];

                    ctx.beginPath();
                    ctx.moveTo(hull.vertices[0].x, hull.vertices[0].y);
                    for(let i=1; i<hull.vertices.length; i++) {{
                        ctx.lineTo(hull.vertices[i].x, hull.vertices[i].y);
                    }}
                    ctx.closePath();

                    // Fill
                    ctx.fillStyle = `rgba(${{color[0]}}, ${{color[1]}}, ${{color[2]}}, 0.05)`;
                    ctx.fill();

                    // Stroke
                    ctx.strokeStyle = `rgba(${{color[0]}}, ${{color[1]}}, ${{color[2]}}, 0.6)`;
                    ctx.lineWidth = 2;
                    ctx.stroke();
                }});

                // Draw points
                projected.forEach(p => {{
                    const r = Math.max(2, 4 * p.scale);
                    if (r < 0.5) return;

                    const color = clusterColors[p.cluster % clusterColors.length];
                    const depthAlpha = Math.max(0.3, Math.min(1, (p.z + 1.5) / 2));

                    ctx.beginPath();
                    ctx.arc(p.x, p.y, r, 0, Math.PI * 2);
                    ctx.fillStyle = `rgba(${{color[0]}}, ${{color[1]}}, ${{color[2]}}, ${{depthAlpha}})`;
                    ctx.fill();

                    ctx.strokeStyle = `rgba(${{color[0]}}, ${{color[1]}}, ${{color[2]}}, ${{depthAlpha * 0.5}})`;
                    ctx.lineWidth = 0.5;
                    ctx.stroke();
                }});

                canvas.projected = projected;
            }}

            // Interaction handlers
            wrapper.addEventListener('wheel', e => {{
                e.preventDefault();
                const zoomSpeed = 0.0015;
                const factor = Math.exp(-e.deltaY * zoomSpeed);
                state.scale *= factor;
                state.scale = Math.max(0.1, Math.min(100.0, state.scale));
                requestAnimationFrame(render);
            }}, {{ passive: false }});

            canvas.addEventListener('mousedown', e => {{
                e.preventDefault();
                isDragging = true;
                dragButton = e.button;
                lastPos = {{ x: e.clientX, y: e.clientY }};
                wrapper.style.cursor = (dragButton === 0) ? 'grabbing' : 'move';
            }});

            canvas.addEventListener('contextmenu', e => e.preventDefault());

            window.addEventListener('mouseup', () => {{
                isDragging = false;
                dragButton = null;
                wrapper.style.cursor = 'default';
            }});

            window.addEventListener('mousemove', e => {{
                if(isDragging) {{
                    const dx = e.clientX - lastPos.x;
                    const dy = e.clientY - lastPos.y;

                    if (dragButton === 0) {{
                        state.rotationY += dx * 0.005;
                        state.rotationX += dy * 0.005;
                    }} else if (dragButton === 2 || dragButton === 1) {{
                        state.panX += dx;
                        state.panY += dy;
                    }}

                    lastPos = {{ x: e.clientX, y: e.clientY }};
                    requestAnimationFrame(render);
                }} else {{
                    // Tooltip
                    const rect = canvas.getBoundingClientRect();
                    const mx = e.clientX - rect.left;
                    const my = e.clientY - rect.top;

                    if (!canvas.projected) return;

                    let hit = null;
                    for (let i = canvas.projected.length - 1; i >= 0; i--) {{
                        const p = canvas.projected[i];
                        const dist = (p.x - mx)**2 + (p.y - my)**2;
                        const hitRadius = Math.max(25, (10 * p.scale)**2);

                        if (dist < hitRadius) {{
                            hit = p;
                            break;
                        }}
                    }}

                    if(hit) {{
                        tooltip.style.opacity = 1;
                        tooltip.style.left = (e.clientX + 20) + 'px';
                        tooltip.style.top = e.clientY + 'px';
                        tooltip.innerHTML = `<div class="cluster-id">Cluster ${{hit.cluster}}</div>Point ID: ${{hit.original.id}}`;
                        canvas.style.cursor = 'pointer';
                    }} else {{
                        tooltip.style.opacity = 0;
                        canvas.style.cursor = 'default';
                    }}
                }}
            }});

            resize();
        </script>
    </body>
    </html>
    """

    return HTML(html)


# USE IT:
viz = create_gnn_cluster_3d_viz(
    projections=projections,
    cluster_labels=results['labels'],
    hull_vertices=results['hull_vertices'],
    height=850
)

viz


In [ ]:
from IPython.display import HTML
import json
import numpy as np

def create_gnn_cluster_3d_viz(projections, cluster_labels, hull_vertices, terms, height=850):
    """
    D3.js 3D scatter plot for GNN clustering with convex hull envelopes.
    Text only appears on hover.
    """

    # Clean Data
    clean_proj = np.nan_to_num(projections, nan=0.0, posinf=0.0, neginf=0.0)

    # Prepare point data
    data = []
    for i in range(len(cluster_labels)):
        data.append({
            'x': float(clean_proj[i, 0]),
            'y': float(clean_proj[i, 1]),
            'z': float(clean_proj[i, 2]),
            'cluster': int(cluster_labels[i]),
            'term': str(terms[i]).replace('"', '\\"'),
            'id': i
        })

    # Prepare hull data
    hulls = []
    for cluster_id, hull_verts in hull_vertices.items():
        hull_clean = np.nan_to_num(hull_verts, nan=0.0, posinf=0.0, neginf=0.0)
        hull_points = []
        for point in hull_clean:
            hull_points.append({
                'x': float(point[0]),
                'y': float(point[1]),
                'z': float(point[2])
            })
        hulls.append({
            'cluster': int(cluster_id),
            'vertices': hull_points
        })

    unique_clusters = list(set([int(label) for label in cluster_labels]))
    n_clusters = len(unique_clusters)

    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <script src="https://d3js.org/d3.v7.min.js"></script>
        <style>
            body, html {{ margin: 0; padding: 0; overflow: hidden; }}

            #viz-wrapper {{
                position: relative;
                width: 100%;
                height: {height}px;
                background: radial-gradient(circle at center, #f8f9fa 0%, #e0e7ff 100%);
                border: 1px solid #ddd;
                border-radius: 8px;
                cursor: default;
            }}

            #main-canvas {{
                display: block;
                width: 100%;
                height: 100%;
            }}

            .overlay {{
                position: absolute;
                top: 20px;
                left: 20px;
                pointer-events: none;
                z-index: 10;
                background: rgba(255, 255, 255, 0.85);
                padding: 12px 18px;
                border-radius: 8px;
                backdrop-filter: blur(4px);
                border: 1px solid rgba(0,0,0,0.05);
            }}

            .title {{
                font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
                font-size: 18px;
                font-weight: 700;
                color: #2c3e50;
                margin: 0;
            }}

            .subtitle {{
                font-family: sans-serif;
                font-size: 12px;
                color: #666;
                margin-top: 4px;
            }}

            .controls-hint {{
                position: absolute;
                bottom: 20px;
                left: 50%;
                transform: translateX(-50%);
                background: rgba(0,0,0,0.7);
                color: white;
                padding: 8px 16px;
                border-radius: 20px;
                font-size: 12px;
                font-family: sans-serif;
                pointer-events: none;
                opacity: 0.6;
            }}

            .tooltip {{
                position: absolute;
                background: rgba(30, 30, 35, 0.95);
                color: #fff;
                padding: 12px 16px;
                border-radius: 8px;
                font-family: sans-serif;
                font-size: 13px;
                line-height: 1.4;
                pointer-events: none;
                opacity: 0;
                transition: opacity 0.1s;
                z-index: 100;
                box-shadow: 0 10px 25px rgba(0,0,0,0.2);
            }}

            .cluster-id {{ font-weight: 700; color: #60a5fa; margin-bottom: 4px; font-size: 14px; }}
        </style>
    </head>
    <body>
        <div id="viz-wrapper">
            <canvas id="main-canvas"></canvas>

            <div class="overlay">
                <div class="title">GNN Clustering (3D)</div>
                <div class="subtitle">{len(data)} points • {n_clusters} clusters</div>
            </div>

            <div class="controls-hint">
                🖱️ Left: Rotate &nbsp; • &nbsp; 🖱️ Right: Pan &nbsp; • &nbsp; 🖱️ Scroll: Zoom
            </div>

            <div id="tooltip" class="tooltip"></div>
        </div>

        <script>
            const data = {json.dumps(data)};
            const hulls = {json.dumps(hulls)};

            const wrapper = document.getElementById('viz-wrapper');
            const canvas = document.getElementById('main-canvas');
            const ctx = canvas.getContext('2d');
            const tooltip = document.getElementById('tooltip');

            const clusterColors = [
                [59, 130, 246], [239, 68, 68], [34, 197, 94], [234, 179, 8], [168, 85, 247],
                [236, 72, 153], [20, 184, 166], [249, 115, 22], [14, 165, 233], [132, 204, 22]
            ];

            let state = {{ rotationX: 0.5, rotationY: 0.5, scale: 1.0, distance: 2.5, panX: 0.0, panY: 0.0 }};
            let isDragging = false, dragButton = null, lastPos = {{ x: 0, y: 0 }}, width, height;

            const xScale = d3.scaleLinear().domain(d3.extent(data, d => d.x)).range([-1, 1]);
            const yScale = d3.scaleLinear().domain(d3.extent(data, d => d.y)).range([-1, 1]);
            const zScale = d3.scaleLinear().domain(d3.extent(data, d => d.z)).range([-1, 1]);

            function resize() {{
                width = wrapper.clientWidth;
                height = wrapper.clientHeight;
                const dpr = window.devicePixelRatio || 1;
                canvas.width = width * dpr;
                canvas.height = height * dpr;
                canvas.style.width = width + 'px';
                canvas.style.height = height + 'px';
                ctx.scale(dpr, dpr);
                requestAnimationFrame(render);
            }}
            window.addEventListener('resize', resize);

            function project3D(x, y, z) {{
                const cx = Math.cos(state.rotationX), sx = Math.sin(state.rotationX);
                const cy = Math.cos(state.rotationY), sy = Math.sin(state.rotationY);
                const x1 = x * cy + z * sy, z1 = -x * sy + z * cy;
                const y1 = y * cx - z1 * sx, z2 = y * sx + z1 * cx;
                const cameraZ = z2 + state.distance;
                if (cameraZ <= 0.1) return null;
                const baseSize = Math.min(width, height) * 0.4, perspective = 1 / cameraZ;
                return {{
                    x: width/2 + (x1 * perspective * baseSize * state.scale) + state.panX,
                    y: height/2 - (y1 * perspective * baseSize * state.scale) + state.panY,
                    z: z2, scale: perspective * state.scale
                }};
            }}

            function render() {{
                ctx.clearRect(0, 0, width, height);
                const projected = [], hullProjected = [];

                data.forEach(d => {{
                    const p = project3D(xScale(d.x), yScale(d.y), zScale(d.z));
                    if (p) projected.push({{ ...p, cluster: d.cluster, term: d.term, original: d }});
                }});

                hulls.forEach(hull => {{
                    const hullVerts = hull.vertices.map(v => project3D(xScale(v.x), yScale(v.y), zScale(v.z))).filter(p => p);
                    if (hullVerts.length >= 3) {{
                        hullProjected.push({{
                            cluster: hull.cluster,
                            vertices: hullVerts,
                            avgZ: hullVerts.reduce((s, v) => s + v.z, 0) / hullVerts.length
                        }});
                    }}
                }});

                projected.sort((a, b) => b.z - a.z);
                hullProjected.sort((a, b) => b.avgZ - a.avgZ);

                hullProjected.forEach(hull => {{
                    const color = clusterColors[hull.cluster % clusterColors.length];
                    ctx.beginPath();
                    ctx.moveTo(hull.vertices[0].x, hull.vertices[0].y);
                    hull.vertices.slice(1).forEach(v => ctx.lineTo(v.x, v.y));
                    ctx.closePath();
                    ctx.fillStyle = `rgba(${{color[0]}}, ${{color[1]}}, ${{color[2]}}, 0.05)`;
                    ctx.fill();
                    ctx.strokeStyle = `rgba(${{color[0]}}, ${{color[1]}}, ${{color[2]}}, 0.6)`;
                    ctx.lineWidth = 2;
                    ctx.stroke();
                }});

                projected.forEach(p => {{
                    const r = Math.max(2, 4 * p.scale);
                    if (r < 0.5) return;
                    const color = clusterColors[p.cluster % clusterColors.length];
                    const alpha = Math.max(0.3, Math.min(1, (p.z + 1.5) / 2));
                    ctx.beginPath();
                    ctx.arc(p.x, p.y, r, 0, Math.PI * 2);
                    ctx.fillStyle = `rgba(${{color[0]}}, ${{color[1]}}, ${{color[2]}}, ${{alpha}})`;
                    ctx.fill();
                    ctx.strokeStyle = `rgba(${{color[0]}}, ${{color[1]}}, ${{color[2]}}, ${{alpha * 0.5}})`;
                    ctx.lineWidth = 0.5;
                    ctx.stroke();
                }});

                canvas.projected = projected;
            }}

            wrapper.addEventListener('wheel', e => {{
                e.preventDefault();
                state.scale *= Math.exp(-e.deltaY * 0.0015);
                state.scale = Math.max(0.1, Math.min(100.0, state.scale));
                requestAnimationFrame(render);
            }}, {{ passive: false }});

            canvas.addEventListener('mousedown', e => {{
                e.preventDefault();
                isDragging = true;
                dragButton = e.button;
                lastPos = {{ x: e.clientX, y: e.clientY }};
                wrapper.style.cursor = (dragButton === 0) ? 'grabbing' : 'move';
            }});

            canvas.addEventListener('contextmenu', e => e.preventDefault());
            window.addEventListener('mouseup', () => {{ isDragging = false; dragButton = null; wrapper.style.cursor = 'default'; }});

            window.addEventListener('mousemove', e => {{
                if(isDragging) {{
                    const dx = e.clientX - lastPos.x, dy = e.clientY - lastPos.y;
                    if (dragButton === 0) {{ state.rotationY += dx * 0.005; state.rotationX += dy * 0.005; }}
                    else if (dragButton === 2 || dragButton === 1) {{ state.panX += dx; state.panY += dy; }}
                    lastPos = {{ x: e.clientX, y: e.clientY }};
                    requestAnimationFrame(render);
                }} else {{
                    const rect = canvas.getBoundingClientRect();
                    const mx = e.clientX - rect.left, my = e.clientY - rect.top;
                    if (!canvas.projected) return;

                    let hit = null;
                    for (let i = canvas.projected.length - 1; i >= 0; i--) {{
                        const p = canvas.projected[i];
                        if ((p.x - mx)**2 + (p.y - my)**2 < Math.max(25, (10 * p.scale)**2)) {{
                            hit = p;
                            break;
                        }}
                    }}

                    if(hit) {{
                        tooltip.style.opacity = 1;
                        tooltip.style.left = (e.clientX + 20) + 'px';
                        tooltip.style.top = e.clientY + 'px';
                        tooltip.innerHTML = `<div class="cluster-id">Cluster ${{hit.cluster}}</div><strong>${{hit.term}}</strong><br>Point ID: ${{hit.original.id}}`;
                        canvas.style.cursor = 'pointer';
                    }} else {{
                        tooltip.style.opacity = 0;
                        canvas.style.cursor = 'default';
                    }}
                }}
            }});

            resize();
        </script>
    </body>
    </html>
    """

    return HTML(html)


# USE IT:
viz = create_gnn_cluster_3d_viz(
    projections=projections,
    cluster_labels=results['labels'],
    hull_vertices=results['hull_vertices'],
    terms=terms,
    height=850
)

viz


## breaking

In [ ]:
import re
from nltk.stem import WordNetLemmatizer

def preprocess_definition(definition):
    # Remove punctuation, lowercase
    text = re.sub(r'[^\w\s]', ' ', definition.lower())

    # Tokenize and lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in text.split()]

    return tokens


In [ ]:
def build_term_index(data):
    term_to_id = {}
    id_to_term = {}

    for tid, entry in data.items():
        term_normalized = entry['term'].lower()
        term_to_id[term_normalized] = tid
        id_to_term[tid] = entry['term']

        # Also index individual words in multi-word terms
        words = term_normalized.split()
        for word in words:
            if word not in term_to_id:
                term_to_id[word] = []
            if isinstance(term_to_id[word], list):
                term_to_id[word].append(tid)

    return term_to_id, id_to_term


In [ ]:
def extract_edges(data, term_to_id):
    edges = []

    for source_id, entry in data.items():
        definition = entry['definition']
        tokens = preprocess_definition(definition)

        # Find which tokens are terms
        referenced_terms = set()

        for token in tokens:
            if token in term_to_id:
                target_ids = term_to_id[token]
                if isinstance(target_ids, str):
                    target_ids = [target_ids]

                for target_id in target_ids:
                    if target_id != source_id:  # No self-loops
                        referenced_terms.add(target_id)

        # Create edges with weights
        for target_id in referenced_terms:
            weight = tokens.count(id_to_term[target_id].lower())
            edges.append({
                'source': source_id,
                'target': target_id,
                'weight': weight
            })

    return edges


In [ ]:
import json
import re
from collections import defaultdict

# YOUR DATA HERE (replace with your actual JSON)
data = {
    "01": {
        "term": "heat capacity",
        "definition": "Heat absorbed (or released) by a system per unit of temperature rise (or fall)."
    },
    "02": {
        "term": "ablation",
        "definition": "1) Combined processes (such as melting, sublimation, evaporation or calving) which remove snow or ice from a glacier or from a snowfield; also used to express the quantity lost by these processes. 2) Reduction of the water equivalent of snow cover by melting, evaporation, wind and avalanches."
    },
    "07": {
        "term": "thermal capacity",
        "definition": "Thermal capacity is the ability of a substance to absorb heat without undergoing a change in its temperature. It is defined as the amount of heat required to raise the temperature of one unit mass of a substance by one degree."
    },
    "08": {
        "term": "lethal concentration",
        "definition": "Concentration of a toxic substance in water at which at least 50 per cent of any living organisms are killed after a specified exposure time."
    }
}


def preprocess_text(text):
    """Clean and tokenize text"""
    text = re.sub(r'\(\d+\)', '', text)  # Remove (1), (2)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    return text.split()


def extract_edges(data):
    """Extract directed edges between terms"""

    stopwords = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'been', 'be',
                 'have', 'has', 'had', 'do', 'does', 'will', 'of', 'to', 'for',
                 'in', 'on', 'at', 'by', 'from', 'with', 'as', 'or', 'and', 'but',
                 'unit', 'degree', 'amount', 'given', 'least', 'cent', 'time'}

    # Build term vocabulary
    term_to_id = {}
    for tid, entry in data.items():
        term = entry['term'].lower().strip()
        term_to_id[term] = tid

        # Index individual words
        words = preprocess_text(term)
        for word in words:
            if len(word) > 3:
                if word not in term_to_id:
                    term_to_id[word] = []
                if isinstance(term_to_id[word], list):
                    term_to_id[word].append(tid)
                else:
                    term_to_id[word] = [term_to_id[word], tid]

    # Find edges
    edges = []
    edge_details = defaultdict(lambda: {'weight': 0, 'words': set()})

    for source_id, entry in data.items():
        def_tokens = preprocess_text(entry['definition'])
        token_counts = defaultdict(int)

        for token in def_tokens:
            if token not in stopwords and len(token) > 3:
                token_counts[token] += 1

        for token, count in token_counts.items():
            if token in term_to_id:
                target_ids = term_to_id[token]
                if not isinstance(target_ids, list):
                    target_ids = [target_ids]

                for target_id in target_ids:
                    if target_id != source_id:
                        key = (source_id, target_id)
                        edge_details[key]['weight'] += count
                        edge_details[key]['words'].add(token)

    for (source, target), details in edge_details.items():
        if details['weight'] >= 1:
            edges.append({
                'source': source,
                'target': target,
                'weight': details['weight'],
                'words': list(details['words'])
            })

    return edges


# RUN THIS
print("Extracting edges...")
edges = extract_edges(data)

print(f"\n✓ Found {len(edges)} edges\n")
for edge in edges[:10]:  # Show first 10
    src = data[edge['source']]['term']
    tgt = data[edge['target']]['term']
    print(f"{src:25} → {tgt:25} | Weight: {edge['weight']}")


Extracting edges...

✓ Found 1 edges

thermal capacity          → heat capacity             | Weight: 3


In [ ]:
from IPython.display import HTML
import json
import numpy as np

def create_gnn_cluster_3d_viz(projections, cluster_labels, hull_vertices, terms, edges=None, height=850):
    """
    D3.js 3D scatter plot for GNN clustering with convex hull envelopes and directed edges.
    Text only appears on hover.
    """

    # Clean Data
    clean_proj = np.nan_to_num(projections, nan=0.0, posinf=0.0, neginf=0.0)

    # Prepare point data
    data = []
    for i in range(len(cluster_labels)):
        data.append({
            'x': float(clean_proj[i, 0]),
            'y': float(clean_proj[i, 1]),
            'z': float(clean_proj[i, 2]),
            'cluster': int(cluster_labels[i]),
            'term': str(terms[i]).replace('"', '\\"'),
            'id': i
        })

    # Prepare hull data
    hulls = []
    for cluster_id, hull_verts in hull_vertices.items():
        hull_clean = np.nan_to_num(hull_verts, nan=0.0, posinf=0.0, neginf=0.0)
        hull_points = []
        for point in hull_clean:
            hull_points.append({
                'x': float(point[0]),
                'y': float(point[1]),
                'z': float(point[2])
            })
        hulls.append({
            'cluster': int(cluster_id),
            'vertices': hull_points
        })

    # Prepare edge data (if provided)
    links = []
    if edges is not None:
        for edge in edges:
            try:
                # Adjust indexing based on your data
                src_id = int(edge['source']) - 1  # Adjust if IDs start at 1
                tgt_id = int(edge['target']) - 1

                if 0 <= src_id < len(data) and 0 <= tgt_id < len(data):
                    links.append({
                        'source': src_id,
                        'target': tgt_id,
                        'weight': edge.get('weight', 1)
                    })
            except (ValueError, KeyError):
                continue

    unique_clusters = list(set([int(label) for label in cluster_labels]))
    n_clusters = len(unique_clusters)

    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <script src="https://d3js.org/d3.v7.min.js"></script>
        <style>
            body, html {{ margin: 0; padding: 0; overflow: hidden; }}
            #viz-wrapper {{ position: relative; width: 100%; height: {height}px;
                background: radial-gradient(circle at center, #f8f9fa 0%, #e0e7ff 100%);
                border: 1px solid #ddd; border-radius: 8px; cursor: default; }}
            #main-canvas {{ display: block; width: 100%; height: 100%; }}
            .overlay {{ position: absolute; top: 20px; left: 20px; pointer-events: none; z-index: 10;
                background: rgba(255, 255, 255, 0.85); padding: 12px 18px; border-radius: 8px;
                backdrop-filter: blur(4px); border: 1px solid rgba(0,0,0,0.05); }}
            .title {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
                font-size: 18px; font-weight: 700; color: #2c3e50; margin: 0; }}
            .subtitle {{ font-family: sans-serif; font-size: 12px; color: #666; margin-top: 4px; }}
            .controls-hint {{ position: absolute; bottom: 20px; left: 50%; transform: translateX(-50%);
                background: rgba(0,0,0,0.7); color: white; padding: 8px 16px; border-radius: 20px;
                font-size: 12px; font-family: sans-serif; pointer-events: none; opacity: 0.6; }}
            .tooltip {{ position: absolute; background: rgba(30, 30, 35, 0.95); color: #fff;
                padding: 12px 16px; border-radius: 8px; font-family: sans-serif; font-size: 13px;
                line-height: 1.4; pointer-events: none; opacity: 0; transition: opacity 0.1s;
                z-index: 100; box-shadow: 0 10px 25px rgba(0,0,0,0.2); }}
            .cluster-id {{ font-weight: 700; color: #60a5fa; margin-bottom: 4px; font-size: 14px; }}
        </style>
    </head>
    <body>
        <div id="viz-wrapper">
            <canvas id="main-canvas"></canvas>
            <div class="overlay">
                <div class="title">GNN Clustering (3D)</div>
                <div class="subtitle">{len(data)} points • {n_clusters} clusters • {len(links)} edges</div>
            </div>
            <div class="controls-hint">🖱️ Left: Rotate • Right: Pan • Scroll: Zoom</div>
            <div id="tooltip" class="tooltip"></div>
        </div>
        <script>
            const data = {json.dumps(data)};
            const hulls = {json.dumps(hulls)};
            const links = {json.dumps(links)};
            const wrapper = document.getElementById('viz-wrapper');
            const canvas = document.getElementById('main-canvas');
            const ctx = canvas.getContext('2d');
            const tooltip = document.getElementById('tooltip');
            const clusterColors = [[59,130,246],[239,68,68],[34,197,94],[234,179,8],[168,85,247],
                [236,72,153],[20,184,166],[249,115,22],[14,165,233],[132,204,22]];
            let state = {{rotationX:0.5,rotationY:0.5,scale:1.0,distance:2.5,panX:0,panY:0}};
            let isDragging=false,dragButton=null,lastPos={{x:0,y:0}},width,height;
            const xScale=d3.scaleLinear().domain(d3.extent(data,d=>d.x)).range([-1,1]);
            const yScale=d3.scaleLinear().domain(d3.extent(data,d=>d.y)).range([-1,1]);
            const zScale=d3.scaleLinear().domain(d3.extent(data,d=>d.z)).range([-1,1]);
            function resize(){{width=wrapper.clientWidth;height=wrapper.clientHeight;const dpr=window.devicePixelRatio||1;
                canvas.width=width*dpr;canvas.height=height*dpr;canvas.style.width=width+'px';
                canvas.style.height=height+'px';ctx.scale(dpr,dpr);requestAnimationFrame(render);}}
            window.addEventListener('resize',resize);
            function project3D(x,y,z){{const cx=Math.cos(state.rotationX),sx=Math.sin(state.rotationX);
                const cy=Math.cos(state.rotationY),sy=Math.sin(state.rotationY);
                const x1=x*cy+z*sy,z1=-x*sy+z*cy;const y1=y*cx-z1*sx,z2=y*sx+z1*cx;
                const cameraZ=z2+state.distance;if(cameraZ<=0.1)return null;
                const baseSize=Math.min(width,height)*0.4,perspective=1/cameraZ;
                return{{x:width/2+(x1*perspective*baseSize*state.scale)+state.panX,
                    y:height/2-(y1*perspective*baseSize*state.scale)+state.panY,z:z2,scale:perspective*state.scale}};}}
            function drawArrow(x1,y1,x2,y2,weight,alpha){{ctx.beginPath();ctx.moveTo(x1,y1);ctx.lineTo(x2,y2);
                ctx.strokeStyle=`rgba(100,100,100,${{alpha*0.4}})`;ctx.lineWidth=Math.max(0.5,Math.min(3,weight*0.5));
                ctx.stroke();const angle=Math.atan2(y2-y1,x2-x1);const arrowSize=8;ctx.beginPath();ctx.moveTo(x2,y2);
                ctx.lineTo(x2-arrowSize*Math.cos(angle-Math.PI/6),y2-arrowSize*Math.sin(angle-Math.PI/6));
                ctx.lineTo(x2-arrowSize*Math.cos(angle+Math.PI/6),y2-arrowSize*Math.sin(angle+Math.PI/6));
                ctx.closePath();ctx.fillStyle=`rgba(100,100,100,${{alpha*0.5}})`;ctx.fill();}}
            function render(){{ctx.clearRect(0,0,width,height);const projected=[],hullProjected=[],edgesProjected=[];
                data.forEach((d,i)=>{{const p=project3D(xScale(d.x),yScale(d.y),zScale(d.z));
                    if(p)projected.push({{...p,cluster:d.cluster,term:d.term,original:d,index:i}});}});
                links.forEach(link=>{{const srcData=data[link.source];const tgtData=data[link.target];
                    const p1=project3D(xScale(srcData.x),yScale(srcData.y),zScale(srcData.z));
                    const p2=project3D(xScale(tgtData.x),yScale(tgtData.y),zScale(tgtData.z));
                    if(p1&&p2){{const avgZ=(p1.z+p2.z)/2;edgesProjected.push({{x1:p1.x,y1:p1.y,z1:p1.z,
                        x2:p2.x,y2:p2.y,z2:p2.z,avgZ:avgZ,weight:link.weight}});}}}});
                hulls.forEach(hull=>{{const hullVerts=hull.vertices.map(v=>project3D(xScale(v.x),yScale(v.y),zScale(v.z))).filter(p=>p);
                    if(hullVerts.length>=3)hullProjected.push({{cluster:hull.cluster,vertices:hullVerts,
                        avgZ:hullVerts.reduce((s,v)=>s+v.z,0)/hullVerts.length}});}});
                projected.sort((a,b)=>b.z-a.z);hullProjected.sort((a,b)=>b.avgZ-a.avgZ);edgesProjected.sort((a,b)=>b.avgZ-a.avgZ);
                hullProjected.forEach(hull=>{{const color=clusterColors[hull.cluster%clusterColors.length];ctx.beginPath();
                    ctx.moveTo(hull.vertices[0].x,hull.vertices[0].y);hull.vertices.slice(1).forEach(v=>ctx.lineTo(v.x,v.y));
                    ctx.closePath();ctx.fillStyle=`rgba(${{color[0]}},${{color[1]}},${{color[2]}},0.05)`;ctx.fill();
                    ctx.strokeStyle=`rgba(${{color[0]}},${{color[1]}},${{color[2]}},0.6)`;ctx.lineWidth=2;ctx.stroke();}});
                edgesProjected.forEach(edge=>{{const alpha=Math.max(0.2,Math.min(1,(edge.avgZ+1.5)/2));
                    drawArrow(edge.x1,edge.y1,edge.x2,edge.y2,edge.weight,alpha);}});
                projected.forEach(p=>{{const r=Math.max(2,4*p.scale);if(r<0.5)return;
                    const color=clusterColors[p.cluster%clusterColors.length];
                    const alpha=Math.max(0.3,Math.min(1,(p.z+1.5)/2));ctx.beginPath();ctx.arc(p.x,p.y,r,0,Math.PI*2);
                    ctx.fillStyle=`rgba(${{color[0]}},${{color[1]}},${{color[2]}},${{alpha}})`;ctx.fill();
                    ctx.strokeStyle=`rgba(${{color[0]}},${{color[1]}},${{color[2]}},${{alpha*0.5}})`;ctx.lineWidth=0.5;ctx.stroke();}});
                canvas.projected=projected;}}
            wrapper.addEventListener('wheel',e=>{{e.preventDefault();state.scale*=Math.exp(-e.deltaY*0.0015);
                state.scale=Math.max(0.1,Math.min(100.0,state.scale));requestAnimationFrame(render);}},{{passive:false}});
            canvas.addEventListener('mousedown',e=>{{e.preventDefault();isDragging=true;dragButton=e.button;
                lastPos={{x:e.clientX,y:e.clientY}};wrapper.style.cursor=(dragButton===0)?'grabbing':'move';}});
            canvas.addEventListener('contextmenu',e=>e.preventDefault());
            window.addEventListener('mouseup',()=>{{isDragging=false;dragButton=null;wrapper.style.cursor='default';}});
            window.addEventListener('mousemove',e=>{{if(isDragging){{const dx=e.clientX-lastPos.x,dy=e.clientY-lastPos.y;
                if(dragButton===0){{state.rotationY+=dx*0.005;state.rotationX+=dy*0.005;}}
                else if(dragButton===2||dragButton===1){{state.panX+=dx;state.panY+=dy;}}
                lastPos={{x:e.clientX,y:e.clientY}};requestAnimationFrame(render);}}else{{
                const rect=canvas.getBoundingClientRect();const mx=e.clientX-rect.left,my=e.clientY-rect.top;
                if(!canvas.projected)return;let hit=null;
                for(let i=canvas.projected.length-1;i>=0;i--){{const p=canvas.projected[i];
                    if((p.x-mx)**2+(p.y-my)**2<Math.max(25,(10*p.scale)**2)){{hit=p;break;}}}}
                if(hit){{tooltip.style.opacity=1;tooltip.style.left=(e.clientX+20)+'px';tooltip.style.top=e.clientY+'px';
                    tooltip.innerHTML=`<div class="cluster-id">Cluster ${{hit.cluster}}</div><strong>${{hit.term}}</strong><br>Point ID: ${{hit.original.id}}`;
                    canvas.style.cursor='pointer';}}else{{tooltip.style.opacity=0;canvas.style.cursor='default';}}}}}});
            resize();
        </script>
    </body>
    </html>
    """

    return HTML(html)


# USAGE:
viz = create_gnn_cluster_3d_viz(
    projections=projections,
    cluster_labels=results['labels'],
    hull_vertices=results['hull_vertices'],
    terms=terms,
    edges=edges,  # <-- Pass your extracted edges here
    height=850
)

viz
